# Prediccion de Palabras con N-gramas

Un modelo de n-gramas predice la siguiente palabra basandose en las anteriores. Es la base de los correctores ortograficos y el autocompletado.

## Ejercicio 1: Construccion de bigramas y trigramas

1. Tokeniza un texto en palabras.
2. Genera bigramas (pares consecutivos) y trigramas (trios).
3. Cuenta la frecuencia de cada n-grama.

In [ ]:
import re
from collections import Counter

texto = """
el procesamiento de lenguaje natural es un campo de la inteligencia artificial
el lenguaje natural permite a las maquinas comprender el texto humano
el texto humano contiene patrones que los modelos aprenden
los modelos de lenguaje predicen la siguiente palabra dado el contexto
el contexto de una palabra determina su significado en el lenguaje natural
"""

tokens = re.sub(r'[^a-z\s]', '', texto.lower()).split()
print(f"Total de tokens: {len(tokens)}")

# Bigramas
bigramas = [(tokens[i], tokens[i+1]) for i in range(len(tokens)-1)]
freq_bi = Counter(bigramas)
print("\nTop 10 bigramas:")
for bigrama, freq in freq_bi.most_common(10):
    print(f"  {bigrama[0]} -> {bigrama[1]} : {freq}")

# Trigramas
trigramas = [(tokens[i], tokens[i+1], tokens[i+2]) for i in range(len(tokens)-2)]
freq_tri = Counter(trigramas)
print("\nTop 10 trigramas:")
for tri, freq in freq_tri.most_common(10):
    print(f"  {tri[0]} {tri[1]} -> {tri[2]} : {freq}")


## Ejercicio 2: Prediccion de la siguiente palabra

1. Construye un modelo de bigramas como diccionario.
2. Dada una palabra, predice las 3 palabras mas probables que siguen.
3. Prueba con varias palabras del vocabulario.

In [ ]:
from collections import defaultdict

def construir_modelo(tokens):
    modelo = defaultdict(Counter)
    for i in range(len(tokens) - 1):
        modelo[tokens[i]][tokens[i+1]] += 1
    return modelo

def predecir(modelo, palabra, top_n=3):
    if palabra not in modelo:
        return []
    sucesores = modelo[palabra]
    total = sum(sucesores.values())
    return [(w, round(c/total, 3)) for w, c in sucesores.most_common(top_n)]

modelo_bi = construir_modelo(tokens)

palabras_prueba = ['el', 'lenguaje', 'de', 'los', 'natural']

print(f"{'Palabra':<15} Siguientes mas probables")
print("-" * 55)
for p in palabras_prueba:
    preds = predecir(modelo_bi, p)
    if preds:
        resultado = ", ".join(f"{w}({prob})" for w, prob in preds)
    else:
        resultado = "(no encontrada)"
    print(f"  {p:<13} {resultado}")


## Ejercicio 3: Autocompletado de frases

1. Usa el modelo de bigramas para completar una frase.
2. Dado un inicio, genera una cadena de palabras predichas.
3. Compara el resultado con el texto original.

In [ ]:
import random

def autocompletar(modelo, inicio, longitud=8, aleatorio=False):
    """Genera una cadena de palabras a partir de una palabra inicial."""
    resultado = [inicio]
    actual = inicio
    for _ in range(longitud - 1):
        if actual not in modelo:
            break
        sucesores = modelo[actual]
        if aleatorio:
            # Seleccion ponderada
            palabras = list(sucesores.keys())
            pesos = list(sucesores.values())
            actual = random.choices(palabras, weights=pesos, k=1)[0]
        else:
            # Siempre la mas probable
            actual = sucesores.most_common(1)[0][0]
        resultado.append(actual)
    return ' '.join(resultado)

inicios = ['el', 'los', 'natural', 'lenguaje']

print("Completado deterministico (mas probable):")
for inicio in inicios:
    if inicio in modelo_bi:
        print(f"  '{inicio}' -> {autocompletar(modelo_bi, inicio, longitud=7)}")

print("\nCompletado aleatorio ponderado:")
random.seed(42)
for inicio in inicios:
    if inicio in modelo_bi:
        print(f"  '{inicio}' -> {autocompletar(modelo_bi, inicio, longitud=7, aleatorio=True)}")
